# Quantum Parameter Sensitivity Analysis

This notebook evaluates the **sensitivity of the quantum black-box output** $F(X)$ with respect to each learnable parameter $\theta_i$, using a finite-difference gradient estimator.

For a perturbation $\delta\theta_i$, we measure:
$$S_i^{\infty} = \frac{\|F(X, \theta + \delta_i) - F(X, \theta)\|_\infty}{\delta A}, \qquad S_i^{2} = \frac{\|F(X, \theta + \delta_i) - F(X, \theta)\|_2}{\delta A}$$

A near-zero sensitivity indicates that the corresponding parameter has **no gradient signal** and cannot be learned.

---
**Config :** `quantum_network_config_dudas` | **Estimator :** `GlobalFiniteDifference`

## 1 · Setup

In [3]:
from dataclasses import fields

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
from zeroth.zeroth_order.gradient_estimators import GlobalFiniteDifferenceConfig

from lab.config.quantum_network_config import quantum_network_config_dudas
from lab.sinus_vs_square_hard.data import create_data
from quantum_simulation.parameters_and_constants import QuantumParameters

# ── Matplotlib style ────────────────────────────────────────────────────────
plt.rcParams.update({
    "figure.dpi": 120,
    "font.family": "serif",
    "axes.spines.top": False,
    "axes.spines.right": False,
})

PARAM_NAMES = [f.name for f in fields(QuantumParameters)]
print("Learnable parameters:", PARAM_NAMES)

ImportError: cannot import name 'QuantumParametersConfig' from 'quantum_simulation.parameters_and_constants.quantum_parameters' (G:\Mon Drive\PycharmProjects\Quantum-Learn\quantum_simulation\parameters_and_constants\quantum_parameters.py)

## 2 · Configuration

In [ ]:
# ── Experiment hyper-parameters ─────────────────────────────────────────────
DA = 0.01  # finite-difference step size
NB_PERIODS = 4  # number of signal periods in the test batch

# ── Instantiate objects ──────────────────────────────────────────────────────
quantum_network = quantum_network_config_dudas.instantiate()
gradient_estimator = GlobalFiniteDifferenceConfig(dA=DA).instantiate(
    nb_params=quantum_network.nb_params)

quantum_network_config_dudas.summary()
GlobalFiniteDifferenceConfig(dA=DA).summary()

## 3 · Input Data

In [ ]:
X, Y = create_data(nb_periods=NB_PERIODS)
X_flat = X.ravel()

print(f"Input shape  : {X.shape}  →  flattened: {X_flat.shape}")
print(f"Label shape  : {Y.shape}")
print(f"Class balance: {int(Y.sum())} sinus / {len(Y) - int(Y.sum())} square")

# ── Quick visualisation of the input sequence ────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 2.5))
ax.step(np.arange(len(X_flat)), X_flat, where="mid", lw=1.2, color="steelblue")
ax.set_xlabel("Time step")
ax.set_ylabel("Amplitude")
ax.set_title(f"Input signal — {NB_PERIODS} periods ({NB_PERIODS // 2} sinus + {NB_PERIODS // 2} square)")
plt.tight_layout()
plt.show()

## 4 · Perturbed Forward Passes

> ⚠️ This cell runs **one simulation per parameter + 1 nominal** — it can take several minutes depending on `SIMULATION_RESOLUTION` and `NB_PERIODS`.

In [ ]:
perturbed_F = quantum_network.forward_perturbed(X=X_flat, gradient_estimator=gradient_estimator)

nb_simulations, seq_len, feature_dim = perturbed_F.shape
print(f"perturbed_F shape : {perturbed_F.shape}")
print(f"  ├─ {nb_simulations} simulations (1 nominal + {nb_simulations - 1} perturbed)")
print(f"  ├─ {seq_len} time steps")
print(f"  └─ {feature_dim} features (quadratures × measure_resolution)")

## 5 · Sensitivity Analysis

In [ ]:
# ── Compute finite differences ───────────────────────────────────────────────
F_nominal = perturbed_F[0]  # shape (seq_len, feature_dim)
dF = perturbed_F[1:] - F_nominal  # shape (nb_params, seq_len, feature_dim)

S_inf = np.max(np.abs(dF), axis=(1, 2)) / DA  # L-infinity norm
S_2 = np.sqrt(np.sum(dF ** 2, axis=(1, 2))) / DA  # L-2 norm

# ── Summary table ────────────────────────────────────────────────────────────
print(f"{'Parameter':<16}  {'S_inf':>10}  {'S_2':>10}  {'Gradient signal?':>18}")
print("-" * 60)
for name, si, s2 in zip(PARAM_NAMES, S_inf, S_2):
    signal = "✔  active" if si > 1e-6 else "✘  dead"
    print(f"{name:<16}  {si:>10.4f}  {s2:>10.4f}  {signal:>18}")

active_params = [n for n, s in zip(PARAM_NAMES, S_inf) if s > 1e-6]
dead_params = [n for n, s in zip(PARAM_NAMES, S_inf) if s <= 1e-6]
print(f"\nActive : {active_params}")
print(f"Dead   : {dead_params}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=False)
colors = ["#e74c3c" if s <= 1e-6 else "#2980b9" for s in S_inf]

for ax, values, label in zip(axes, [S_inf, S_2], [r"$S^\infty$", r"$S^2$"]):
    bars = ax.bar(PARAM_NAMES, values, color=colors, edgecolor="white", linewidth=0.8)
    ax.bar_label(bars, fmt="%.3f", padding=3, fontsize=8)
    ax.set_title(f"Sensitivity — {label} norm / dA", fontsize=11)
    ax.set_xlabel("Parameter")
    ax.set_ylabel(label)
    ax.set_yscale("symlog", linthresh=1e-3)
    ax.yaxis.set_minor_formatter(mticker.NullFormatter())
    ax.tick_params(axis="x", rotation=25)

# Legend
from matplotlib.patches import Patch

axes[0].legend(handles=[
    Patch(color="#2980b9", label="Active (learnable)"),
    Patch(color="#e74c3c", label="Dead (no gradient signal)"),
], fontsize=9, framealpha=0.5)

fig.suptitle("Quantum Black-Box — Parameter Sensitivity", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig("sensitivity_analysis.pdf", bbox_inches="tight")
plt.show()
print("Figure saved → sensitivity_analysis.pdf")

## 6 · Feature-Level Sensitivity Heatmap

Which **features** (quadrature × time-step) are most affected by each parameter?

In [ ]:
# dF_norm shape : (nb_params, seq_len, feature_dim)
dF_abs = np.abs(dF) / DA  # per-feature absolute sensitivity

# Collapse over sequence axis → (nb_params, feature_dim)
dF_feature = dF_abs.mean(axis=1)

fig, ax = plt.subplots(figsize=(12, 3.5))
im = ax.imshow(dF_feature, aspect="auto", cmap="viridis",
               interpolation="nearest")
ax.set_yticks(range(len(PARAM_NAMES)))
ax.set_yticklabels(PARAM_NAMES)
ax.set_xlabel("Feature index")
ax.set_title("Mean absolute sensitivity per parameter and feature (time-averaged)")
plt.colorbar(im, ax=ax, label=r"$|\Delta F| / \delta A$")
plt.tight_layout()
plt.show()

## 7 · Conclusion

| Criterion | Value |
|-----------|-------|
| Perturbation step $\delta A$ | `DA` |
| Number of periods | `NB_PERIODS` |
| Feature dimension | `feature_dim` |
| Active parameters | `active_params` |
| Dead parameters | `dead_params` |

Parameters with $S^\infty \approx 0$ produce **flat loss landscapes** along their direction — the zeroth-order optimiser cannot move them. Consider:
- Fixing dead parameters and removing them from the search space,
- Changing the `encoding_observable` to expose a different coupling,
- Increasing `dA` to check for numerical cancellation.